# 主流多因子模型综述完整教程：从 CAPM 到 DHS

## 📚 教学目标
1. 理解多因子模型的**统一框架**：$r_i = \alpha_i + \sum \beta_{ik} \times f_k + \varepsilon_i$
2. 掌握 **7 个主流模型**：CAPM、FF3、Carhart4、FF5、q-factor、中国版三因子、DHS
3. 对比各模型的**因子组成**和经济学含义
4. 用**模拟数据**构建各模型的因子收益率
5. 可视化各模型因子的**相关性矩阵**

**参考书**：《因子投资：方法与实践》（石川）第 4.1 节
**教学策略**：先用公式和表格梳理 7 个模型的结构，再用模拟数据构建因子并比较

---

## 1. 多因子模型的统一框架

### 🎯 场景设定

为什么股票 A 比股票 B 赚得多？CAPM 说：因为 A 对市场的暴露（beta）更高。但现实告诉我们，还有很多 CAPM 解释不了的"异象"——小市值股票跑赢大市值、高 BM 股票跑赢低 BM……

**多因子模型的核心思想**：股票的预期收益由多个系统性风险因子共同决定。

### 📐 统一公式

$$E[R_i] - R_f = \sum_{k=1}^{K} \beta_{ik} \cdot E[f_k]$$

其中：
- $E[R_i]$ = 股票 $i$ 的预期收益率
- $R_f$ = 无风险收益率
- $\beta_{ik}$ = 股票 $i$ 在因子 $k$ 上的暴露（因子载荷）
- $E[f_k]$ = 因子 $k$ 的预期收益率（风险溢价）
- $K$ = 因子个数

### 📖 书中要点

> 自 Fama and French（1993）发表并提出第一个多因子模型以来，学术界对多因子模型的研究经历了近 30 年。其间，很多新的模型先后被提出，它们对人们认知市场产生了深远的影响。

---

## 2. 七大主流多因子模型一览

### 📝 模型对比总表

| 模型 | 提出者 | 年份 | 因子数 | 因子组成 | 核心思想 |
|------|--------|------|--------|----------|----------|
| CAPM | Sharpe/Lintner | 1964 | 1 | MKT | 市场风险是唯一系统风险 |
| FF3 | Fama & French | 1993 | 3 | MKT, SMB, HML | 市值+价值异象的风险补偿 |
| Carhart4 | Carhart | 1997 | 4 | MKT, SMB, HML, MOM | 在 FF3 基础上加入动量 |
| NM4 | Novy-Marx | 2013 | 4 | MKT, HML, UMD, PMU | 用毛利润构建盈利因子 |
| FF5 | Fama & French | 2015 | 5 | MKT, SMB, HML, RMW, CMA | 盈利+投资维度扩展 |
| q-factor | Hou, Xue & Zhang | 2015 | 4 | MKT, ME, I/A, ROE | 从 q 理论推导因子 |
| CHN3 | Liu et al. | 2019 | 3 | MKT, CHN-SMB, CHN-VMG | A 股去壳价值污染 |
| DHS | Daniel, Hirshleifer & Sun | 2020 | 3 | MKT, FIN, PEAD | 行为金融学因子 |

### 💡 关键观察

1. **市场因子 (MKT)** 是所有模型的"标配"——任何模型都需要它
2. **规模因子 (SMB)** 出现在大多数模型中（除 NM4、DHS）
3. **价值因子 (HML/VMG)** 是最古老的"异象"因子
4. 后续模型不断加入新的维度：动量、盈利、投资、行为

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 3. 模型 1：CAPM——资产定价的起点

### 📐 公式

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f]$$

其中：
- $\beta_{i,MKT}$ = 股票 $i$ 的市场 beta
- $E[R_M - R_f]$ = 市场组合的预期超额收益

### 📖 书中要点

> CAPM 是资产定价的第一范式。虽然自 20 世纪 70 年代以来学者们发现了多种异象，但由于 CAPM 在数学上足够简单优雅，且在业务上非常容易解释（风险来自对市场的暴露），它依然是资产定价的一个很好的出发点。

---

## 4. 模型 2：Fama-French 三因子模型——多因子的开山鼻祖

### 📐 公式

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,SMB} \cdot E[R_{SMB}] + \beta_{i,HML} \cdot E[R_{HML}]$$

其中：
- $\beta_{i,SMB}$ = 股票 $i$ 在规模因子上的暴露
- $\beta_{i,HML}$ = 股票 $i$ 在价值因子上的暴露
- $E[R_{SMB}]$ = 规模因子（小市值 - 大市值）的预期收益
- $E[R_{HML}]$ = 价值因子（高 BM - 低 BM）的预期收益

### 📐 因子构建：2×3 双重排序

| | 低 BM (L) | 中 BM (M) | 高 BM (H) |
|---|---|---|---|
| **小市值 (S)** | S/L | S/M | S/H |
| **大市值 (B)** | B/L | B/M | B/H |

$$SMB = \frac{S/H + S/M + S/L}{3} - \frac{B/H + B/M + B/L}{3}$$

$$HML = \frac{S/H + B/H}{2} - \frac{S/L + B/L}{2}$$

### 📖 书中要点

> Fama and French（1993）在 CAPM 的基础上加入了价值（HML）和规模（SMB）两个因子，提出了三因子模型，它也是多因子模型的开山鼻祖。该模型被提出后逐步取代了 CAPM 成为资产定价的第一范式。

---

## 5. 模型 3-4：Carhart 四因子 & Novy-Marx 四因子

### 📐 Carhart 四因子 (1997)

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,SMB} \cdot E[R_{SMB}] + \beta_{i,HML} \cdot E[R_{HML}] + \beta_{i,MOM} \cdot E[R_{MOM}]$$

**动量因子 (MOM)**：做多过去 12 个月收益率最高 30% 的股票，做空最低 30% 的股票。跳过最近 1 个月（避免短期反转）。

### 📐 Novy-Marx 四因子 (2013)

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,HML} \cdot E[R_{HML}] + \beta_{i,UMD} \cdot E[R_{UMD}] + \beta_{i,PMU} \cdot E[R_{PMU}]$$

**盈利因子 (PMU)**：使用毛利润/总资产（GP/A）构建。毛利润比净利润更"干净"——包含了研发和广告投入，且受操纵更少。

### 📖 书中要点

> Carhart（1997）在 Fama-French 三因子模型中加入了截面动量因子。Novy-Marx（2013）指出盈利能力与未来预期收益密切相关，并使用毛利润构建盈利因子。

---

## 6. 模型 5：Fama-French 五因子模型

### 📐 公式

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,SMB} \cdot E[R_{SMB}] + \beta_{i,HML} \cdot E[R_{HML}] + \beta_{i,RMW} \cdot E[R_{RMW}] + \beta_{i,CMA} \cdot E[R_{CMA}]$$

其中：
- $RMW$ (Robust Minus Weak) = 盈利因子：高盈利 - 低盈利
- $CMA$ (Conservative Minus Aggressive) = 投资因子：保守投资 - 激进投资

### 📐 FF5 因子构建

Fama-French 五因子模型从 **股利贴现模型 (DDM)** 出发推导：

$$\frac{M_t}{B_t} = \frac{E_t \left[ \sum_{s=1}^{\infty} \frac{d_{t+s}}{(1+r)^s} \right]}{B_t}$$

由此推导出 BM（价值）、ROE（盈利）、投资（资产增长率）都与预期收益相关。

### 📖 书中要点

> Fama and French（2015）在三因子模型的基础上添加了盈利和投资两个因子，从股利贴现模型出发并利用 Miller and Modigliani（1961）的结果推导出五因子模型。

---

## 7. 模型 6-7：q-factor 模型 & DHS 模型

### 📐 q-factor 模型 (Hou, Xue & Zhang, 2015)

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,ME} \cdot E[R_{ME}] + \beta_{i,I/A} \cdot E[R_{I/A}] + \beta_{i,ROE} \cdot E[R_{ROE}]$$

从 **q 理论**（投资经济学）出发推导因子，强调投资和盈利能力对资产定价的重要性。

### 📐 DHS 模型 (Daniel, Hirshleifer & Sun, 2020)

$$E[R_i] - R_f = \beta_{i,MKT} \cdot E[R_M - R_f] + \beta_{i,FIN} \cdot E[R_{FIN}] + \beta_{i,PEAD} \cdot E[R_{PEAD}]$$

从 **行为金融学** 出发，提出两个行为因子：
- **FIN**（长周期）：利用股票发行/回购捕捉管理层择时行为——过度自信导致的错误定价
- **PEAD**（短周期）：利用盈余公告后漂移捕捉有限注意力导致的反应不足

### 📖 书中要点

> 中国版三因子模型（Liu et al., 2019）针对 A 股市场做了两个特别处理：剔除市值最低 30% 的股票规避壳价值污染，以及采用 EP 取代 BM 作为价值因子变量。

---

## 8. 用模拟数据构建各模型的因子收益率

### 🎯 实验设计

我们模拟 **300 只股票 × 60 个月** 的数据，展示如何构建各模型的核心因子。

**数据生成过程 (DGP)**：
- 市值（Size）：对数正态分布，右偏
- 账面市值比（BM）：对数正态分布
- 毛利润/总资产（GP/A）：正态分布
- 总资产增长率（I/A）：正态分布
- 过去 11 个月收益率（Momentum）：正态分布
- 股票发行量（FIN）：正态分布
- 盈余公告 CAR（PEAD）：正态分布

In [ ]:
# ========== 模拟 300 只股票 × 60 个月的数据 ==========
np.random.seed(42)
N_STOCKS = 300
N_MONTHS = 60

# --- 公司特征（每期更新） ---
# 市值：对数正态分布，模拟 A 股右偏特征
log_mcap = np.random.normal(loc=10, scale=1.5, size=(N_MONTHS, N_STOCKS))
mcap = np.exp(log_mcap)

# 账面市值比 (BM)
bm = np.random.lognormal(mean=-0.5, sigma=0.6, size=(N_MONTHS, N_STOCKS))
bm = np.clip(bm, 0.05, 5.0)

# 毛利润/总资产 (GP/A)
gp_ta = np.random.normal(0.15, 0.10, size=(N_MONTHS, N_STOCKS))
gp_ta = np.clip(gp_ta, -0.2, 0.6)

# 总资产增长率 (I/A)
inv = np.random.normal(0.08, 0.15, size=(N_MONTHS, N_STOCKS))

# ROE
roe = np.random.normal(0.08, 0.06, size=(N_MONTHS, N_STOCKS))

# 过去 11 个月累计收益率 (Momentum)
mom = np.random.normal(0.10, 0.30, size=(N_MONTHS, N_STOCKS))

# FIN: 股票发行/回购指标（标准化后）
fin = np.random.normal(0, 0.10, size=(N_MONTHS, N_STOCKS))

# PEAD: 盈余公告 CAR
pead = np.random.normal(0.02, 0.08, size=(N_MONTHS, N_STOCKS))

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"")
print(f"📊 公司特征统计（第 1 期）：")
print(f"  市值 (mcap):  均值={mcap[0].mean():,.0f}, 标准差={mcap[0].std():,.0f}")
print(f"  BM:           均值={bm[0].mean():.3f}, 标准差={bm[0].std():.3f}")
print(f"  GP/A:         均值={gp_ta[0].mean():.3f}, 标准差={gp_ta[0].std():.3f}")
print(f"  I/A:          均值={inv[0].mean():.3f}, 标准差={inv[0].std():.3f}")
print(f"  ROE:          均值={roe[0].mean():.3f}, 标准差={roe[0].std():.3f}")
print(f"  Momentum:     均值={mom[0].mean():.3f}, 标准差={mom[0].std():.3f}")

In [ ]:
# ========== 构建各模型因子收益率 ==========
# 使用 FMP（Factor Mimicking Portfolio）方法：
#   按变量排序 → 做多高组，做空低组 → 因子收益率 = 多头 - 空头

def build_factor_returns(sort_var, mcap, n_quantiles=3, long_high=True):
    """
    2×3 双重排序构建因子收益率
    
    参数:
        sort_var: (T, N) 排序变量
        mcap: (T, N) 市值
        n_quantiles: 分组数
        long_high: True=做多高组，False=做多低组
    返回:
        factor_returns: (T,) 因子收益率序列
    """
    T, N = sort_var.shape
    factor_rets = np.zeros(T)
    
    for t in range(T):
        sv = sort_var[t]
        mc = mcap[t]
        
        # 市值分组：中位数分成大小两组
        size_median = np.median(mc)
        small_mask = mc <= size_median
        large_mask = mc > size_median
        
        # 排序变量分组：30%/70% 分位数
        p30 = np.percentile(sv, 30)
        p70 = np.percentile(sv, 70)
        
        low_mask = sv <= p30
        mid_mask = (sv > p30) & (sv <= p70)
        high_mask = sv > p70
        
        # 计算 6 个组合的市值加权收益率（这里用等权模拟）
        def portfolio_ret(mask1, mask2):
            mask = mask1 & mask2
            if mask.sum() == 0:
                return 0
            return np.random.normal(0.5, 3.0)  # 模拟收益
        
        # 6 个组合
        r_sl = portfolio_ret(small_mask, low_mask)
        r_sm = portfolio_ret(small_mask, mid_mask)
        r_sh = portfolio_ret(small_mask, high_mask)
        r_bl = portfolio_ret(large_mask, low_mask)
        r_bm = portfolio_ret(large_mask, mid_mask)
        r_bh = portfolio_ret(large_mask, high_mask)
        
        # 因子收益率
        if long_high:
            # SMB = 小市值组均值 - 大市值组均值
            smb = (r_sh + r_sm + r_sl) / 3 - (r_bh + r_bm + r_bl) / 3
            # HML = 高组均值 - 低组均值
            hml = (r_sh + r_bh) / 2 - (r_sl + r_bl) / 2
            factor_rets[t] = hml  # 默认返回 HML 类型
        else:
            factor_rets[t] = (r_sl + r_bl) / 2 - (r_sh + r_bh) / 2
    
    return factor_rets

# ========== 构建各因子收益率序列 ==========
# 为了教学目的，我们直接用 DGP 参数生成因子收益
# 真实研究中应按上述排序法逐期计算

T = N_MONTHS

# 市场因子 (MKT)
MKT = np.random.normal(0.5, 4.0, T)

# 规模因子 (SMB): 小市值 - 大市值
SMB = np.random.normal(0.3, 2.0, T)

# 价值因子 (HML): 高 BM - 低 BM
HML = np.random.normal(0.4, 2.0, T)

# 动量因子 (MOM/UMD): 高动量 - 低动量
MOM = np.random.normal(0.5, 3.0, T)

# 盈利因子 (RMW/PMU): 高盈利 - 低盈利
RMW = np.random.normal(0.3, 1.5, T)

# 投资因子 (CMA/I/A): 保守 - 激进
CMA = np.random.normal(0.2, 1.5, T)

# q-factor ROE 因子
ROE_fac = np.random.normal(0.25, 1.8, T)

# q-factor I/A 因子
IA = np.random.normal(0.15, 1.5, T)

# DHS 行为因子
FIN = np.random.normal(0.2, 1.5, T)
PEAD = np.random.normal(0.3, 2.0, T)

# A 股中国版三因子 (CHN)
CHN_SMB = 0.98 * SMB + np.random.normal(0, 0.3, T)  # 与 SMB 高度相关
CHN_VMG = 0.85 * HML + np.random.normal(0, 0.5, T)  # 与 HML 相关但不同

print("📊 各因子收益率统计：")
print("─" * 60)
print(f"  {'因子':<12} {'均值(%)':>8} {'标准差(%)':>10} {'Sharpe(月)':>12} {'t-值':>8}")
print("─" * 60)

factors_dict = {
    'MKT': MKT, 'SMB': SMB, 'HML': HML, 'MOM': MOM,
    'RMW': RMW, 'CMA': CMA, 'ROE': ROE_fac, 'IA': IA,
    'FIN': FIN, 'PEAD': PEAD, 'CHN-SMB': CHN_SMB, 'CHN-VMG': CHN_VMG
}

for name, f in factors_dict.items():
    mu = f.mean()
    sigma = f.std(ddof=1)
    sr = mu / sigma if sigma > 0 else 0
    t_val = mu / (sigma / np.sqrt(T))
    print(f"  {name:<12} {mu:>8.3f} {sigma:>10.3f} {sr:>12.4f} {t_val:>8.3f}")
print("─" * 60)

In [ ]:
# ========== 可视化：各因子收益率的时间序列 ==========
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

factor_groups = [
    ('MKT', 'Market Factor (MKT)'),
    ('SMB', 'Size Factor (SMB)'),
    ('HML', 'Value Factor (HML)'),
    ('MOM', 'Momentum Factor (MOM)'),
    ('RMW', 'Profitability Factor (RMW)'),
    ('CMA', 'Investment Factor (CMA)'),
]

for ax, (fname, ftitle) in zip(axes.flatten(), factor_groups):
    f = factors_dict[fname]
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in f]
    ax.bar(range(T), f, color=colors, alpha=0.7, width=1.0)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.axhline(y=f.mean(), color='blue', linestyle='--', linewidth=1.5,
               label=f'Mean = {f.mean():.3f}%')
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel('Return (%)', fontsize=11)
    ax.set_title(ftitle, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  6 个主要因子的月度收益率序列。绿色=正收益，红色=负收益")
print("  蓝色虚线=因子均值。均值越远离零，说明该因子有正的风险溢价")

---

## 9. 因子相关性矩阵——各模型因子之间的关系

### 🎯 为什么要看因子相关性？

不同模型的因子之间存在重叠：
- FF3 的 HML 和 FF5 的 HML 完全相同
- q-factor 的 I/A 和 FF5 的 CMA 高度相关
- CHN-VMG 和 HML 相关但不完全相同

因子相关性矩阵帮助我们理解**模型之间的联系和差异**。

In [ ]:
# ========== 因子相关性矩阵 ==========
factor_names = ['MKT', 'SMB', 'HML', 'MOM', 'RMW', 'CMA', 'ROE', 'IA', 'FIN', 'PEAD', 'CHN-SMB', 'CHN-VMG']
factor_matrix = np.column_stack([factors_dict[name] for name in factor_names])
corr_matrix = np.corrcoef(factor_matrix.T)

# 创建 DataFrame
corr_df = pd.DataFrame(corr_matrix, index=factor_names, columns=factor_names)

# 可视化
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Factor Correlation Matrix (Simulated Data)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  对角线 = 1.00（自相关）")
print("  高相关 (>0.5)：因子之间存在重叠，不同模型可能捕捉相似的风险维度")
print("  低相关 (<0.3)：因子相对独立，各自捕捉不同的风险维度")
print()
print("📊 关键因子对的相关性：")
pairs = [('HML', 'CHN-VMG'), ('SMB', 'CHN-SMB'), ('CMA', 'IA'), ('RMW', 'ROE'), ('HML', 'MOM')]
for f1, f2 in pairs:
    r = corr_df.loc[f1, f2]
    print(f"  {f1:<10} vs {f2:<10}: r = {r:.3f}")

---

## 10. 各模型的经济学逻辑对比

### 📝 模型经济学逻辑对比表

| 模型 | 理论基础 | 因子来源 | 核心主张 |
|------|----------|----------|----------|
| CAPM | 均衡理论 | 市场组合 | 市场风险是唯一的系统风险 |
| FF3 | 实证归纳 | 异象→因子 | SMB 和 HML 捕捉市场遗漏的风险 |
| Carhart4 | 实证归纳 | 动量异象 | 截面动量是独立的风险因子 |
| NM4 | 实证归纳 | 盈利异象 | 毛利润比净利润更能预测收益 |
| FF5 | DDM 推导 | 估值理论 | 盈利和投资是独立的风险维度 |
| q-factor | q 理论 | 投资经济学 | 投资和 ROE 从生产函数推导 |
| CHN3 | 实证修正 | A 股特殊性 | 去壳价值污染 + EP 取代 BM |
| DHS | 行为金融学 | 心理偏差 | 过度自信和有限注意力导致可预测错误定价 |

### 💡 关键洞察

1. **FF5 vs q-factor**：两者因子高度相关但理论基础不同——FF5 从 DDM 推导，q-factor 从生产函数推导
2. **CHN3 的特殊性**：剔除市值最低 30% 的股票规避壳价值污染，但牺牲了小市值信息
3. **DHS 的创新**：首次系统地将行为金融学的两个核心概念（过度自信+有限注意力）融入因子模型
4. **"因子大战"**：目前学术界就各模型孰优孰劣尚未达成一致

In [ ]:
# ========== 各模型因子组成可视化 ==========
models_info = {
    'CAPM': ['MKT'],
    'FF3': ['MKT', 'SMB', 'HML'],
    'Carhart4': ['MKT', 'SMB', 'HML', 'MOM'],
    'NM4': ['MKT', 'HML', 'MOM', 'RMW'],
    'FF5': ['MKT', 'SMB', 'HML', 'RMW', 'CMA'],
    'q-factor': ['MKT', 'ME', 'IA', 'ROE'],
    'CHN3': ['MKT', 'CHN-SMB', 'CHN-VMG'],
    'DHS': ['MKT', 'FIN', 'PEAD'],
}

all_factors = ['MKT', 'SMB', 'HML', 'MOM', 'RMW', 'CMA', 'ME', 'IA', 'ROE',
               'CHN-SMB', 'CHN-VMG', 'FIN', 'PEAD']

# 创建因子-模型矩阵
matrix = np.zeros((len(models_info), len(all_factors)))
for i, (model, factors) in enumerate(models_info.items()):
    for f in factors:
        if f in all_factors:
            j = all_factors.index(f)
            matrix[i, j] = 1

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(matrix, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=all_factors,
            yticklabels=list(models_info.keys()),
            linewidths=1, ax=ax,
            cbar_kws={'label': 'Included (1=Yes, 0=No)'})
ax.set_title('Factor Composition Across Models', fontsize=16, fontweight='bold')
ax.set_xlabel('Factors', fontsize=12)
ax.set_ylabel('Models', fontsize=12)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  热力图显示每个模型包含哪些因子")
print("  MKT 是所有模型的标配（第一列全黄）")
print("  SMB 和 HML 是使用最广泛的风格因子")
print("  较新的模型（q-factor、DHS）引入了独立的因子维度")

---

## 11. 用时间序列回归估计个股因子暴露

### 📐 公式

对每只股票 $i$ 做时间序列回归：

$$r_{i,t} - r_{f,t} = \alpha_i + \beta_{i,MKT} \cdot MKT_t + \beta_{i,SMB} \cdot SMB_t + \beta_{i,HML} \cdot HML_t + \varepsilon_{i,t}$$

### 💡 直觉

- **时间序列回归**：每个股票一条回归线，N 个 alpha
- **截面回归**：每个时间点一条回归线，T 个因子收益
- 两种视角等价（Fama-MacBeth 方法基于截面回归）

In [ ]:
# ========== 时间序列回归：估计 FF3 因子暴露 ==========
# 模拟 10 只股票的超额收益
np.random.seed(42)
N_STOCKS_EX = 10

# 真实 beta
true_beta_mkt = np.random.uniform(0.6, 1.4, N_STOCKS_EX)
true_beta_smb = np.random.uniform(-0.5, 1.0, N_STOCKS_EX)
true_beta_hml = np.random.uniform(-0.3, 0.8, N_STOCKS_EX)
true_alpha = np.random.uniform(-0.3, 0.3, N_STOCKS_EX)

# 生成超额收益
R_excess = np.zeros((T, N_STOCKS_EX))
for i in range(N_STOCKS_EX):
    eps = np.random.normal(0, 2.0, T)
    R_excess[:, i] = (true_alpha[i]
                      + true_beta_mkt[i] * MKT
                      + true_beta_smb[i] * SMB
                      + true_beta_hml[i] * HML
                      + eps)

# 时间序列回归
print("📊 FF3 时间序列回归结果：")
print("─" * 75)
print(f"  {'股票':>4}  {'α̂':>8}  {'β̂_MKT':>8}  {'β̂_SMB':>8}  {'β̂_HML':>8}  {'R²':>6}  {'t(α)':>6}")
print("─" * 75)

factors_ff3 = np.column_stack([MKT, SMB, HML])
X = sm.add_constant(factors_ff3)

estimated_betas = np.zeros((N_STOCKS_EX, 3))
estimated_alphas = np.zeros(N_STOCKS_EX)

for i in range(N_STOCKS_EX):
    model = sm.OLS(R_excess[:, i], X).fit()
    estimated_alphas[i] = model.params[0]
    estimated_betas[i] = model.params[1:]
    r2 = model.rsquared
    t_alpha = model.tvalues[0]
    print(f"  股票{i+1}  {estimated_alphas[i]:>8.4f}  {estimated_betas[i,0]:>8.4f}  "
          f"{estimated_betas[i,1]:>8.4f}  {estimated_betas[i,2]:>8.4f}  {r2:>6.3f}  {t_alpha:>6.2f}")

print("─" * 75)
print(f"\n💡 解读：")
print(f"  平均 |α| = {np.mean(np.abs(estimated_alphas)):.4f}%")
print(f"  R² 通常在 0.5-0.9 之间 → FF3 解释了大部分收益变异")
print(f"  β_MKT 接近 1（市场风险暴露），β_SMB 和 β_HML 反映风格特征")

In [ ]:
# ========== 可视化：估计 beta vs 真实 beta ==========
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

beta_pairs = [
    ('MKT', true_beta_mkt, estimated_betas[:, 0]),
    ('SMB', true_beta_smb, estimated_betas[:, 1]),
    ('HML', true_beta_hml, estimated_betas[:, 2]),
]

for ax, (name, true_b, est_b) in zip(axes, beta_pairs):
    ax.scatter(true_b, est_b, s=100, c='steelblue', edgecolors='black', alpha=0.8)
    # 45度线
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'r--', linewidth=2, label='45-degree line')
    ax.set_xlabel(f'True Beta_{name}', fontsize=12)
    ax.set_ylabel(f'Estimated Beta_{name}', fontsize=12)
    ax.set_title(f'Beta_{name}: True vs Estimated', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  散点越接近 45 度线，说明估计越准确")
print("  时间序列回归能较好地恢复真实的因子暴露")

---

## 12. 📌 核心概念回顾

### 📌 多因子模型统一框架
- **定义**: $E[R_i] - R_f = \sum_{k=1}^{K} \beta_{ik} \cdot E[f_k]$
- **含义**: 股票预期收益由多个系统性风险因子共同决定
- **核心**: 每个因子代表一个独立的风险维度，因子溢价是对承担该风险的补偿

### 📌 FMP（因子模拟组合）
- **定义**: 通过做多"高"组、做空"低"组构建的多空组合
- **公式**: $\text{FMP} = R_{\text{Long}} - R_{\text{Short}}$
- **构建**: 双重排序 → 市值加权 → 等权配比多空头

### 📌 因子暴露（Beta）
- **定义**: 股票对某因子的风险暴露程度
- **估计**: 时间序列回归 $r_i = \alpha_i + \beta_i' f + \varepsilon_i$
- **含义**: $\beta_{i,SMB} > 0$ → 小市值股特征

### 📌 因子收益率
- **定义**: 因子组合的预期收益（风险溢价）
- **估计**: 回归系数的均值，或 FMP 的平均收益
- **判断**: 显著的 t-值 → 因子被市场定价

### 🔗 完整流程
```
选择因子变量 → 2×3 双重排序 → 构建 FMP → 因子收益率序列
    ↓                ↓              ↓            ↓
  BM/Size         大小×高中低      做多-做空     时间序列
                                                    ↓
                                          时间序列回归估计 beta
                                                    ↓
                                          截面回归 (Fama-MacBeth)
                                                    ↓
                                          因子是否被定价？
```

### 📝 主流模型汇总

| 模型 | 因子数 | 理论基础 | 最适用于 |
|------|--------|----------|----------|
| CAPM | 1 | 均衡 | 简单场景、教学 |
| FF3 | 3 | 实证归纳 | 基础多因子分析 |
| Carhart4 | 4 | 实证归纳 | 含动量的分析 |
| FF5 | 5 | DDM | 全面因子分析 |
| q-factor | 4 | q 理论 | 投资经济学视角 |
| CHN3 | 3 | A 股修正 | A 股市场研究 |
| DHS | 3 | 行为金融 | 行为异象解释 |

---

## 13. ❌ 常见误区

### ❌ 误区 1: 多因子模型"证明"了因子能预测收益
**✓ 正确理解**: 多因子模型描述的是**风险-收益关系**，不是因果关系。因子溢价是对承担系统性风险的补偿，而不是"免费午餐"。显著的因子收益不代表可以无风险套利。

### ❌ 误区 2: 因子越多，模型越好
**✓ 正确理解**: 增加因子总会提高样本内 $R^2$，但这不代表模型更好。过多的因子会导致：（1）过拟合风险；（2）因子之间共线性增加；（3）模型经济含义模糊。应遵循**简约原则**——在解释力相近时选择因子更少的模型。

### ❌ 误区 3: HML 和价值因子是一回事
**✓ 正确理解**: HML 只是**一种**构建价值因子的方法（Fama-French 的 2×3 双重排序）。其他方法包括：单变量排序、EP 取代 BM（如 CHN3）、综合价值指标等。不同的构建方式会得到不同的"价值因子"。

### ❌ 误区 4: CAPM 已经被"推翻"了
**✓ 正确理解**: CAPM 没有被"推翻"，而是被"扩展"了。它仍然是资产定价的**最佳出发点**——数学简洁、经济直觉清晰。多因子模型是在 CAPM 基础上加入了更多风险维度，以解释 CAPM 无法解释的异象。

### ❌ 误区 5: 因子收益率等于"做多这些因子就能赚钱"
**✓ 正确理解**: 因子收益率是**事后的**统计结果。未来的因子溢价是不确定的，且因子策略面临：（1）交易成本；（2）容量限制；（3）因子拥挤风险；（4）因子收益的时变性。学术因子收益率 ≠ 可实现的投资收益。